# Experiment 3.11: Product-Space Muon -- Global Gauge Fixing via SVD Rebalancing

---

## The Per-Layer vs. Product-Space Conditioning Gap

In earlier experiments (notably Exp 2.6, "Product Kappa Crossover vs. Depth"), we established
that Muon's Newton-Schulz orthogonalisation acts as a **per-layer gauge-fixing** operation:
each update step nudges every individual weight matrix $W_\ell$ toward better conditioning
($\kappa(W_\ell) \to 1$) by projecting the gradient onto the orthogonal manifold.

However, per-layer conditioning is a **necessary but not sufficient** condition for healthy
end-to-end gradient flow. The quantity that actually governs signal propagation through a
deep linear network is the **product matrix**:

$$W_{\mathrm{prod}} = W_L \, W_{L-1} \cdots W_1$$

Even if every layer has $\kappa(W_\ell) \approx 1$, the product $W_{\mathrm{prod}}$ can still
become ill-conditioned. This happens because orthogonal matrices, while individually well-conditioned,
can **align their rotation axes** in ways that amplify certain directions and suppress others
across the full depth of the network. The product condition number:

$$\kappa_{\mathrm{prod}} = \frac{\sigma_{\max}(W_{\mathrm{prod}})}{\sigma_{\min}(W_{\mathrm{prod}})}$$

captures this global spectral health. It is bounded above by $\prod_\ell \kappa(W_\ell)$, but can
also grow through **cross-layer spectral interference** even when per-layer condition numbers are small.

## The Product Rebalancing Fix: From Local to Global Gauge Fixing

This experiment introduces a **product-space rebalancing** operation that extends Muon's
gauge-fixing from the per-layer (local) level to the full-network (global) level.

Every $K$ steps, we:
1. Compute the full product $W_{\mathrm{prod}} = W_L \cdots W_1$
2. SVD-decompose it: $W_{\mathrm{prod}} = U \Sigma V^\top$
3. Redistribute the singular values equally: set $S = \mathrm{diag}(\sigma_i^{1/L})$
4. Rebuild the factorisation: $W_1 = S \, V^\top$, $W_i = S$ for $i \in \{2,\ldots,L-1\}$, $W_L = U \, S$

This preserves the end-to-end map $W_{\mathrm{prod}}$ exactly (the product of the new layers equals
the old product), but **equalises** the singular-value contribution across layers. In the RG language,
this is a **global gauge transformation** that removes the redundant degrees of freedom in how the
total spectral mass is distributed among layers.

## Hypothesis

**(b) prevents the kappa crossover** -- product kappa stays controlled even at depth 8 where
plain Muon struggles. The periodic SVD rebalancing acts as a global gauge-fixing step that
complements Muon's per-layer local gauge fixing, preventing the exponential compounding of
cross-layer spectral misalignment.

## Methodology

- **Architecture:** 8-layer deep linear network, $32 \times 32$ weight matrices
- **Training:** 500 optimisation steps with momentum ($\beta = 0.9$)
- **Target:** Ill-conditioned linear map with $\kappa = 100$ (log-spaced singular values from 100 to 1)
- **Initialisation:** Xavier (Glorot) init, shared across all methods via same random seed
- **Comparison:**
  - **(a) Plain Muon** -- per-layer Newton-Schulz orthogonalisation only
  - **(b) Muon + Product Rebalance** -- SVD rebalancing every $K = 50$ steps
  - **(c) SGD** -- momentum SGD baseline, no spectral intervention
- **Metrics:** Product condition number $\kappa_{\mathrm{prod}} = \sigma_{\max} / \sigma_{\min}$ of $W_{\mathrm{prod}}$ at every step
- **Learning rate:** Per-method grid search over 9 candidates, selecting the LR that minimises loss after 200 steps
- **Seeds:** 5 independent runs for statistical robustness

## Expected Outcomes

| Test | Criterion | Rationale |
|------|-----------|-----------|
| T1 | $\kappa_{\mathrm{final}}^{\mathrm{RB}} < \kappa_{\mathrm{final}}^{\mathrm{Muon}}$ | Rebalancing directly controls product spectrum |
| T2 | Fewer kappa crossovers for Muon+RB | Global gauge fixing prevents spectral drift |
| T3 | $\mathcal{L}_{\mathrm{final}}^{\mathrm{RB}} < \mathcal{L}_{\mathrm{final}}^{\mathrm{Muon}}$ | Better conditioning enables faster convergence |
| T4 | $\mathcal{L}_{\mathrm{final}}^{\mathrm{RB}} < \mathcal{L}_{\mathrm{final}}^{\mathrm{SGD}}$ | Combined local+global gauge fixing beats no gauge fixing |

## Prior Context

The rebalancing operation is a **balanced factorisation**: given $W_{\mathrm{prod}} = U \Sigma V^\top$,
we set each $W_i = \Sigma^{1/L}$ (with appropriate rotations on the first and last layers).
The product $U \cdot S^L \cdot V^\top = U \Sigma V^\top$ is preserved exactly, but each layer now has
condition number $\kappa(W_i) = (\sigma_{\max}/\sigma_{\min})^{1/L}$ -- the $L$-th root of the
product condition number, which is the minimal achievable per-layer conditioning for this product.

---

## 1. Environment Setup

Import NumPy for all linear algebra operations. This experiment is **pure NumPy** -- no
autograd framework is needed because deep linear networks admit closed-form gradients
via the chain rule on matrix products. The absence of nonlinear activations means
$\partial \mathcal{L} / \partial W_\ell$ can be computed analytically as an outer product
of backpropagated error signals and forward activations.

In [ ]:
import numpy as np
import os

---

## 2. Reproducibility and Random Seed

Setting a fixed seed ensures that all random matrix initialisations and data generation
are deterministic. This is critical for a fair three-way comparison: SGD, Muon, and
Muon+Rebalance all start from **identical** weight matrices and see **identical** data.

In [ ]:
SEED = 42
np.random.seed(SEED)

print(f"Random seed set for reproducibility")


---

## 3. Experimental Configuration

### Key Design Choices

- **WIDTH = 32:** Large enough for meaningful singular-value spectra (32 modes), small
  enough that SVD at every step remains fast ($O(n^3) \approx 32{,}768$ flops per SVD).
- **NUM_LAYERS = 8:** Deep enough that the product condition number can compound
  significantly ($\kappa_{\mathrm{prod}} \le \kappa^8$ in the worst case), but shallow
  enough to remain numerically tractable. This is the depth regime where Exp 2.6 showed
  plain Muon beginning to struggle with product conditioning.
- **REBALANCE_EVERY = 50:** The rebalancing interval $K$. Too frequent (small $K$)
  and the optimizer barely moves between rebalancings; too rare (large $K$) and
  cross-layer spectral drift accumulates. $K = 50$ gives 10 rebalancings over 500 steps.
- **NS_ITERS = 5:** Newton-Schulz iterations for orthogonalisation. Converges the
  polar factor to machine precision for $32 \times 32$ matrices (cubic convergence rate).
- **MOMENTUM = 0.9:** Standard heavy-ball momentum. Note that after a rebalancing step,
  velocities are **reset** because the weight-space geometry has changed discontinuously.
- **NUM_SEEDS = 5:** Enough for basic statistical characterisation (mean and std)
  of the stochastic variation across random initialisations.

In [ ]:
WIDTH = 32
NUM_LAYERS = 8
NUM_STEPS = 500
BATCH_SIZE = 64
REBALANCE_EVERY = 50  # K
NS_ITERS = 5
MOMENTUM = 0.9
NUM_SEEDS = 5

print("\n--- Experimental Configuration ---")
print(f"  NUM_LAYERS = {NUM_LAYERS}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  MOMENTUM = {MOMENTUM}")
print(f"  WIDTH = {WIDTH}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  NS_ITERS = {NS_ITERS}")


In [ ]:
# --- Derived quantities for interpretation ---
print("\n--- Derived Quantities ---")
xavier_std = np.sqrt(2.0 / (WIDTH + WIDTH))
print(f"  Xavier init std per layer:      sigma = sqrt(2/(n_in+n_out)) = {xavier_std:.4f}")
print(f"  Expected init kappa per layer:  ~O(1) for random Gaussian matrices")
print(f"  Rebalancings per run:           {NUM_STEPS // REBALANCE_EVERY} (every {REBALANCE_EVERY} steps)")
print(f"  Total SVDs for rebalance:       {NUM_STEPS // REBALANCE_EVERY} product SVDs + {NUM_STEPS} per-step kappa SVDs")
print(f"  Ill-conditioned target kappa:   100 (logspace from 100 to 1 across {WIDTH} modes)")
print(f"  Post-rebalance per-layer kappa: 100^(1/{NUM_LAYERS}) = {100**(1.0/NUM_LAYERS):.4f}")
print(f"  This means rebalancing reduces each layer's kappa from potentially large")
print(f"  values down to ~{100**(1.0/NUM_LAYERS):.2f}, a dramatic spectral compression.")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


In [ ]:
# LR search candidates
LR_CANDIDATES = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]

### Learning Rate Search Grid

Each optimizer gets its own best learning rate, found by grid search over 9 candidates.
This is essential for a fair comparison: Muon's orthogonalised updates have unit spectral
norm, so they can typically tolerate larger step sizes than raw SGD gradients. If we used
the same LR for all methods, we would conflate the optimizer's intrinsic benefit with
a learning-rate artifact (as explored in Exp H6a, the LR Confound Audit).

---

## 4. Weight Initialisation

### Xavier (Glorot) Init: $\sigma = \sqrt{2/(n_{\mathrm{in}} + n_{\mathrm{out}})}$

For square $32 \times 32$ layers this gives $\sigma \approx 0.25$. The Marchenko-Pastur law
tells us that random Gaussian matrices of this size have condition numbers concentrating
around $(\sqrt{n}+1)/(\sqrt{n}-1) \approx 1.6$ for $n = 32$. So the initial product
$\kappa_{\mathrm{prod}}$ is approximately $1.6^8 \approx 43$ -- moderate, and well below
the target's $\kappa = 100$.

**Why this matters:** The experiment starts from a regime where all three methods have
similar, manageable product conditioning. The question is how each method's dynamics
*evolve* this product $\kappa$ over 500 steps of training toward the ill-conditioned target.

In [ ]:
def init_weights(num_layers, width, rng):
    """Xavier init."""
    weights = []
    for _ in range(num_layers):
        std = np.sqrt(2.0 / (width + width))
        W = rng.randn(width, width) * std
        weights.append(W.copy())
    return weights

In [ ]:
def copy_weights(weights):
    return [W.copy() for W in weights]

---

## 5. Deep Linear Network: Forward Pass

A deep linear network computes $f(X) = W_L W_{L-1} \cdots W_1 \, X$. Despite being linear,
these networks are **not** trivially optimisable: the loss landscape over $(W_1, \dots, W_L)$
is non-convex due to the multiplicative coupling between layers. Saddle points proliferate,
and gradient flow is highly sensitive to the singular-value spectra of the individual $W_\ell$.

This makes deep linear networks an ideal testbed for studying how optimisers interact with
conditioning: the **only** source of optimisation difficulty is the spectral geometry of
the weight factorisation, with no confounding effects from nonlinear activations.

In [ ]:
def forward_linear(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

---

## 6. Product Matrix Analysis -- The Central Observable

### Product Matrix $W_{\mathrm{prod}} = W_L \cdots W_1$

The product matrix is the **end-to-end linear map** computed by the network. Its SVD reveals:
- **$\sigma_{\max}$**: the direction of maximum signal amplification through the network
- **$\sigma_{\min}$**: the direction of maximum signal suppression
- **$\kappa_{\mathrm{prod}} = \sigma_{\max}/\sigma_{\min}$**: the ratio of these extremes

A large $\kappa_{\mathrm{prod}}$ means the network amplifies some input directions by orders of
magnitude more than others. For gradient flow, this translates to: gradients backpropagated
along the suppressed directions are exponentially attenuated, creating an effective
"gradient dead zone" that slows learning of those modes.

### Why Track $\kappa_{\mathrm{prod}}$ Instead of Per-Layer $\kappa$?

Muon's per-layer orthogonalisation controls $\kappa(W_\ell)$ for each layer individually.
But the product $\kappa_{\mathrm{prod}}$ can still grow if the **rotational components** of
adjacent layers conspire to align dominant singular directions. This is the "cross-layer
spectral interference" phenomenon that motivates the product rebalancing intervention.

In [ ]:
def compute_product_matrix(weights):
    """Compute W_L @ ... @ W_1."""
    prod = weights[0].copy()
    for W in weights[1:]:
        prod = W @ prod
    return prod

In [ ]:
def product_condition_number(weights):
    """Condition number of the product matrix."""
    prod = compute_product_matrix(weights)
    svs = np.linalg.svd(prod, compute_uv=False)
    if svs[-1] < 1e-15:
        return 1e15
    return svs[0] / svs[-1]

### Loss Function -- Mean Squared Error

$$\mathcal{L} = \frac{1}{2N} \sum_{i=1}^{N} \|f(x_i) - y_i\|^2$$

MSE is the canonical loss for regression. In the deep linear setting it induces a loss
surface whose critical points are fully characterised by the SVD of the target matrix
(Saxe et al., 2014). The *path* the optimiser takes through this landscape -- and whether
it accumulates or sheds ill-conditioning along the way -- is exactly what this experiment measures.

In [ ]:
def compute_loss(weights, X, Y_target):
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

### Backpropagation -- Analytic Gradients

For layer $\ell$, the gradient is $\partial \mathcal{L} / \partial W_\ell = \delta_\ell \, a_{\ell-1}^\top$,
where $\delta_\ell$ is the error signal backpropagated through layers $\ell+1, \dots, L$ and
$a_{\ell-1}$ is the pre-activation at layer $\ell$. The recursion is $\delta_\ell = W_{\ell+1}^\top \delta_{\ell+1}$.

**Connection to conditioning:** If $W_{\ell+1}$ has a large condition number, the
backpropagated signal $\delta_\ell$ gets stretched along the dominant singular direction
and crushed along the smallest. This directional bias in the gradient causes SGD to
*reinforce* existing anisotropy -- a positive feedback loop that drives
$\kappa_{\mathrm{prod}}$ upward.

In [ ]:
def compute_gradients(weights, X, Y_target):
    num_layers = len(weights)
    N = X.shape[1]
    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])
    delta = (activations[-1] - Y_target) / N
    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

---

## 7. Newton-Schulz Orthogonalisation -- The Core of Muon (Local Gauge Fixing)

Given a gradient matrix $G$, Newton-Schulz iteration computes the **polar factor**
$U = G (G^\top G)^{-1/2}$ -- the closest orthogonal matrix to $G$ in Frobenius norm.
The iteration used here is:

$$X_{k+1} = \tfrac{3}{2} X_k - \tfrac{1}{2} X_k (X_k^\top X_k)$$

This converges cubically, so 5 iterations suffice for $32 \times 32$ matrices.

**Why orthogonal updates help conditioning:** An orthogonal matrix has all singular values
equal to 1. Subtracting $\eta U$ from $W$ pushes the singular values of $W$ toward a
narrower band, reducing $\kappa(W)$ per step. SGD subtracts the raw gradient $G$, whose
singular-value spectrum mirrors the *existing* anisotropy of the loss landscape --
reinforcing rather than correcting ill-conditioning.

**Limitation (motivating this experiment):** This orthogonalisation is **per-layer**. It
fixes the local gauge (singular-value spread within each $W_\ell$) but cannot fix the
**global gauge** (how the product of rotations across layers distributes the total
spectral mass). That is the job of the product rebalancing operation defined below.

In [ ]:
def newton_schulz_ortho(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-12:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

---

## 8. Product Rebalancing -- Global Gauge Fixing via SVD Redistribution

### The Algorithm

Every $K$ steps, we perform a **balanced refactorisation** of the deep linear network:

1. **Compute** $W_{\mathrm{prod}} = W_L \cdots W_1$ (the end-to-end map)
2. **SVD** it: $W_{\mathrm{prod}} = U \Sigma V^\top$
3. **Redistribute** singular values: $S = \mathrm{diag}(\sigma_i^{1/L})$
4. **Rebuild** layers: $W_1 = S V^\top$, $W_i = S$ for middle layers, $W_L = U S$

### Why This Preserves the Function

The product of the new layers is:
$$W_L^{\mathrm{new}} \cdots W_1^{\mathrm{new}} = (U S) \cdot S^{L-2} \cdot (S V^\top) = U S^L V^\top = U \Sigma V^\top = W_{\mathrm{prod}}$$

So the network computes exactly the same function before and after rebalancing. This is a
**pure gauge transformation** -- it changes the internal parametrisation without changing
the input-output map.

### The Physical Intuition

Think of each layer's singular values as "energy levels" in a physical system. Without
rebalancing, training can create "hot" layers (large singular values) and "cold" layers
(small singular values). The hot layers dominate gradient flow, creating a positive
feedback loop. Rebalancing thermalises the system by redistributing energy equally,
so every layer carries $\sigma_i^{1/L}$ of each mode's total contribution.

### Velocity Reset After Rebalancing

After rebalancing, the momentum buffers (velocities) are **zeroed** because the
weight-space geometry has changed discontinuously. The old velocity vectors point
in directions that are no longer meaningful in the rebalanced coordinate system.
This is analogous to resetting the "memory" of the optimizer after a coordinate change.

In [ ]:
def rebalance_product(weights):
    """
    Compute product W_prod = W_L ... W_1, SVD it, and redistribute
    singular values equally across layers.

    W_prod = U @ diag(sigma) @ V^T
    Set W_1 = diag(sigma^{1/L}) @ V^T
    Set W_i = diag(sigma^{1/L}) for i in 2..L-1  (with random rotations absorbed)
    Set W_L = U @ diag(sigma^{1/L})

    Actually, a cleaner approach: use the balanced factorization.
    W_prod = U @ diag(sigma) @ V^T
    sigma_balanced = sigma^{1/L}
    W_1 = diag(sigma_balanced) @ V^T
    W_L = U @ diag(sigma_balanced)
    W_i = diag(sigma_balanced) @ Q_i for middle layers (using identity rotations)

    Simplest correct approach that preserves the product:
    W_prod = U @ diag(sigma) @ V^T
    Let S = diag(sigma^{1/L})
    W_1 = S @ V^T
    W_i = S    for i = 2, ..., L-1
    W_L = U @ S
    Product = U @ S @ S @ ... @ S @ V^T = U @ S^L @ V^T = U @ diag(sigma) @ V^T

    But this ignores that S^L = diag(sigma) only if sigma has ALL positive entries.
    Since SVD guarantees sigma >= 0, we're fine.
    """
    L = len(weights)
    prod = compute_product_matrix(weights)
    U, sigma, Vt = np.linalg.svd(prod, full_matrices=True)

    # sigma^{1/L} -- handle zeros
    sigma_balanced = np.power(np.maximum(sigma, 1e-30), 1.0 / L)
    S = np.diag(sigma_balanced)

    new_weights = []
    # Layer 0: S @ V^T
    new_weights.append(S @ Vt)
    # Middle layers: S (diagonal in standard basis)
    for _ in range(1, L - 1):
        new_weights.append(S.copy())
    # Last layer: U @ S
    new_weights.append(U @ S)

    return new_weights

---

## 9. Training Engine -- Unified Loop for All Three Methods

The training function implements all three methods in a single loop:

- **SGD**: $W_\ell \leftarrow W_\ell - \eta (\beta v_\ell + \nabla_{W_\ell} \mathcal{L})$ with momentum $\beta = 0.9$
- **Muon**: Same as SGD but gradients are orthogonalised via Newton-Schulz before the momentum update
- **Muon + Rebalance**: Same as Muon, but every $K$ steps the product matrix is SVD-rebalanced
  and momentum buffers are reset

At every step, we record both the loss $\mathcal{L}$ and the product condition number
$\kappa_{\mathrm{prod}}$. This gives us full trajectories for both observables, enabling
detailed analysis of how conditioning and convergence co-evolve.

**Divergence guard:** If the loss exceeds $10^{10}$ or becomes NaN, the run is terminated
early and padded with sentinel values. This prevents wasting compute on blown-up runs
while preserving the trajectory up to the point of divergence.

In [ ]:
def train(weights_init, Y_target, X, n_steps, method='sgd', lr=0.01,
          rebalance_every=None):
    """
    Train and return losses, product kappas.

    method: 'sgd', 'muon', 'muon_rebalance'
    """
    weights = copy_weights(weights_init)
    velocities = [np.zeros_like(w) for w in weights]

    losses = []
    kappas = []

    for step in range(n_steps):
        loss = compute_loss(weights, X, Y_target)
        kappa = product_condition_number(weights)
        losses.append(loss)
        kappas.append(kappa)

        if np.isnan(loss) or loss > 1e10:
            losses.extend([1e10] * (n_steps - step - 1))
            kappas.extend([1e15] * (n_steps - step - 1))
            break

        grads = compute_gradients(weights, X, Y_target)

        if method == 'sgd':
            for i in range(len(weights)):
                velocities[i] = MOMENTUM * velocities[i] + grads[i]
                weights[i] = weights[i] - lr * velocities[i]
        elif method in ('muon', 'muon_rebalance'):
            for i in range(len(weights)):
                ortho_grad = newton_schulz_ortho(grads[i])
                velocities[i] = MOMENTUM * velocities[i] + ortho_grad
                weights[i] = weights[i] - lr * velocities[i]

            # Product rebalance
            if method == 'muon_rebalance' and rebalance_every is not None:
                if (step + 1) % rebalance_every == 0:
                    weights = rebalance_product(weights)
                    # Reset velocities after rebalance (weight space changed)
                    velocities = [np.zeros_like(w) for w in weights]

    # Final measurements
    losses.append(compute_loss(weights, X, Y_target))
    kappas.append(product_condition_number(weights))

    return np.array(losses), np.array(kappas)

In [ ]:
def find_best_lr(weights_init, Y_target, X, method, rebalance_every=None):
    """Grid search for best LR."""
    best_loss = 1e20
    best_lr = 0.005
    for lr in LR_CANDIDATES:
        w = copy_weights(weights_init)
        losses, _ = train(w, Y_target, X, min(200, NUM_STEPS), method=method,
                          lr=lr, rebalance_every=rebalance_every)
        final = losses[-1]
        if not np.isnan(final) and final < best_loss:
            best_loss = final
            best_lr = lr
    return best_lr

### Learning Rate Grid Search

For each seed and method, we run a short (200-step) training with each candidate LR
and select the one that achieves the lowest final loss. This ensures each method operates
at its optimal step size, eliminating learning-rate mismatch as a confound.

**Why 200 steps for the search?** Long enough to distinguish good LRs from bad ones
(divergent or too slow), short enough to keep the search cheap relative to the full 500-step run.

---

## 10. Main Experiment Execution

### Protocol for Each Seed

For each of the 5 random seeds:
1. **Generate target:** An ill-conditioned linear map $T = U_t \Sigma_t V_t^\top$ with
   $\kappa(T) = 100$. The singular values are log-spaced from 100 down to 1, creating
   a challenging regression target with a 100:1 dynamic range.
2. **Generate data:** Random input $X$ and target $Y = T \cdot X'$ (both scaled by 0.3).
3. **Initialise weights:** Xavier init, shared across all three methods.
4. **LR search:** Find best LR for each method via grid search.
5. **Full training:** Run all three methods for 500 steps, recording loss and $\kappa_{\mathrm{prod}}$.

### What to Watch in the Output

- **Initial product kappa:** Should be moderate (~10-100) from Xavier init at depth 8
- **Selected LRs:** Muon may select larger LRs than SGD (orthogonal updates are scale-normalised)
- **Final kappa comparison:** The key test -- does Muon+RB achieve lower $\kappa$ than plain Muon?
- **Final loss comparison:** Does the conditioning advantage translate to better optimisation?

In [ ]:
print("=" * 85)
print("Experiment 3.11: Product-Space Muon")
print("=" * 85)
print(f"Setup: {NUM_LAYERS}-layer deep linear ({WIDTH}x{WIDTH})")
print(f"Steps: {NUM_STEPS}, Rebalance every: {REBALANCE_EVERY} steps")
print(f"Seeds: {NUM_SEEDS}")
print("=" * 85)

### Results Accumulators

We store full loss and kappa curves for each seed and method. This enables both:
- **Aggregate statistics** (mean/std across seeds) for the final summary
- **Per-seed trajectory analysis** for the crossover detection algorithm

In [ ]:
# Results accumulators
all_results = {
    'sgd': {'final_losses': [], 'final_kappas': [], 'kappa_curves': [], 'loss_curves': []},
    'muon': {'final_losses': [], 'final_kappas': [], 'kappa_curves': [], 'loss_curves': []},
    'muon_rebalance': {'final_losses': [], 'final_kappas': [], 'kappa_curves': [], 'loss_curves': []},
}

### Full Training Runs Across Seeds

This is the core computation. For each seed, we:
1. Generate an ill-conditioned target with $\kappa = 100$
2. Perform per-method LR grid search
3. Run full 500-step training for all three methods from identical init
4. Print per-seed diagnostics: initial kappa, selected LRs, final loss and kappa

**Interpretation guide for the per-seed output:**
- Initial product kappa should be O(10-100) from Xavier init at depth 8
- If Muon+RB selects a different LR from plain Muon, the rebalancing changes the loss landscape geometry
- Compare final kappa across methods: T1 predicts Muon+RB < Muon < SGD
- Compare final loss: T3 and T4 predict Muon+RB achieves the lowest loss

In [ ]:
for seed_idx in range(NUM_SEEDS):
    run_seed = SEED + seed_idx * 17
    rng = np.random.RandomState(run_seed)

    print(f"\n--- Seed {run_seed} ---")

    # Ill-conditioned target
    U_t, _ = np.linalg.qr(rng.randn(WIDTH, WIDTH))
    V_t, _ = np.linalg.qr(rng.randn(WIDTH, WIDTH))
    sigma_t = np.logspace(2, 0, WIDTH)  # 100 down to 1, kappa=100
    W_target = U_t @ np.diag(sigma_t) @ V_t
    Y_target = W_target @ np.random.randn(WIDTH, BATCH_SIZE) * 0.3
    X = np.random.randn(WIDTH, BATCH_SIZE) * 0.3

    # Same init for all methods
    weights_init = init_weights(NUM_LAYERS, WIDTH, rng)
    init_kappa = product_condition_number(weights_init)
    print(f"  Initial product kappa: {init_kappa:.2f}")

    # Find LRs
    lr_sgd = find_best_lr(weights_init, Y_target, X, 'sgd')
    lr_muon = find_best_lr(weights_init, Y_target, X, 'muon')
    lr_muon_rb = find_best_lr(weights_init, Y_target, X, 'muon_rebalance',
                               rebalance_every=REBALANCE_EVERY)
    print(f"  LRs: SGD={lr_sgd}, Muon={lr_muon}, Muon+Rebal={lr_muon_rb}")

    # Full runs
    for method, lr, rb_every in [('sgd', lr_sgd, None),
                                   ('muon', lr_muon, None),
                                   ('muon_rebalance', lr_muon_rb, REBALANCE_EVERY)]:
        losses, kappas = train(copy_weights(weights_init), Y_target, X,
                               NUM_STEPS, method=method, lr=lr,
                               rebalance_every=rb_every)
        all_results[method]['final_losses'].append(losses[-1])
        all_results[method]['final_kappas'].append(kappas[-1])
        all_results[method]['loss_curves'].append(losses)
        all_results[method]['kappa_curves'].append(kappas)

    print(f"  Finals: SGD loss={all_results['sgd']['final_losses'][-1]:.4e} kappa={all_results['sgd']['final_kappas'][-1]:.1f}")
    print(f"          Muon loss={all_results['muon']['final_losses'][-1]:.4e} kappa={all_results['muon']['final_kappas'][-1]:.1f}")
    print(f"          Muon+RB loss={all_results['muon_rebalance']['final_losses'][-1]:.4e} kappa={all_results['muon_rebalance']['final_kappas'][-1]:.1f}")

In [ ]:
# --- Intermediate: Per-seed summary and method ranking ---
print("\n" + "=" * 85)
print("PER-SEED SUMMARY: METHOD RANKING")
print("=" * 85)
print(f"\n{'Seed':>6} {'Best Loss Method':>18} {'Best Kappa Method':>20} {'Muon+RB kappa ratio':>22}")
print("-" * 70)
for seed_idx in range(NUM_SEEDS):
    losses = {m: all_results[m]['final_losses'][seed_idx] for m in ['sgd', 'muon', 'muon_rebalance']}
    kappas = {m: all_results[m]['final_kappas'][seed_idx] for m in ['sgd', 'muon', 'muon_rebalance']}
    best_loss_method = min(losses, key=losses.get)
    best_kappa_method = min(kappas, key=kappas.get)
    # Ratio of Muon+RB kappa to Muon kappa (< 1 means rebalancing helps)
    kappa_ratio = kappas['muon_rebalance'] / max(kappas['muon'], 1e-15)
    label_map = {'sgd': 'SGD', 'muon': 'Muon', 'muon_rebalance': 'Muon+RB'}
    print(f"  {seed_idx:>4} {label_map[best_loss_method]:>18} {label_map[best_kappa_method]:>20} {kappa_ratio:>22.4f}")

# Count wins
loss_wins = {'sgd': 0, 'muon': 0, 'muon_rebalance': 0}
kappa_wins = {'sgd': 0, 'muon': 0, 'muon_rebalance': 0}
for seed_idx in range(NUM_SEEDS):
    losses = {m: all_results[m]['final_losses'][seed_idx] for m in ['sgd', 'muon', 'muon_rebalance']}
    kappas = {m: all_results[m]['final_kappas'][seed_idx] for m in ['sgd', 'muon', 'muon_rebalance']}
    loss_wins[min(losses, key=losses.get)] += 1
    kappa_wins[min(kappas, key=kappas.get)] += 1

print(f"\n  Loss wins:  SGD={loss_wins['sgd']}, Muon={loss_wins['muon']}, Muon+RB={loss_wins['muon_rebalance']}")
print(f"  Kappa wins: SGD={kappa_wins['sgd']}, Muon={kappa_wins['muon']}, Muon+RB={kappa_wins['muon_rebalance']}")
print(f"\n  Interpretation: Muon+RB kappa ratio < 1 means rebalancing improved conditioning.")
print(f"  A ratio >> 1 would mean rebalancing HURT conditioning (unexpected).")

---

## 11. Aggregate Results -- Statistical Summary Across Seeds

### Mean and Standard Deviation of Final Metrics

The table below summarises the final loss and product condition number for each method,
averaged over all 5 seeds. Key comparisons:

- **Mean Loss:** Lower is better. T3 predicts Muon+RB < Muon; T4 predicts Muon+RB < SGD.
- **Mean Kappa:** Lower is better. T1 predicts Muon+RB < Muon.
- **Std across seeds:** Small std indicates robust results; large std suggests seed sensitivity.

If Muon+RB achieves both lower mean kappa AND lower mean loss than plain Muon, this
constitutes strong evidence that global gauge fixing (product rebalancing) provides a
genuine benefit beyond local gauge fixing (per-layer orthogonalisation).

In [ ]:
print(f"\n\n{'=' * 85}")
print("AGGREGATE RESULTS")
print(f"{'=' * 85}")

In [ ]:
print(f"\n{'Method':<25} {'Mean Loss':>14} {'Std Loss':>14} {'Mean kappa':>14} {'Std kappa':>14}")
print("-" * 85)

In [ ]:
for method in ['sgd', 'muon', 'muon_rebalance']:
    mean_loss = np.mean(all_results[method]['final_losses'])
    std_loss = np.std(all_results[method]['final_losses'])
    mean_kappa = np.mean(all_results[method]['final_kappas'])
    std_kappa = np.std(all_results[method]['final_kappas'])

    label = {'sgd': 'SGD', 'muon': 'Muon (per-layer)',
             'muon_rebalance': f'Muon + Rebalance(K={REBALANCE_EVERY})'}[method]
    print(f"{label:<25} {mean_loss:>14.6e} {std_loss:>14.6e} {mean_kappa:>14.1f} {std_kappa:>14.1f}")

In [ ]:
# --- Intermediate: Effect sizes and relative improvements ---
print("\n--- Effect Size Analysis ---")
mean_loss_sgd = np.mean(all_results['sgd']['final_losses'])
mean_loss_muon = np.mean(all_results['muon']['final_losses'])
mean_loss_rb = np.mean(all_results['muon_rebalance']['final_losses'])
mean_kappa_sgd = np.mean(all_results['sgd']['final_kappas'])
mean_kappa_muon = np.mean(all_results['muon']['final_kappas'])
mean_kappa_rb = np.mean(all_results['muon_rebalance']['final_kappas'])

print(f"\n  Loss reduction (Muon+RB vs Muon):    {(1 - mean_loss_rb/mean_loss_muon)*100:+.1f}%")
print(f"  Loss reduction (Muon+RB vs SGD):      {(1 - mean_loss_rb/mean_loss_sgd)*100:+.1f}%")
print(f"  Loss reduction (Muon vs SGD):          {(1 - mean_loss_muon/mean_loss_sgd)*100:+.1f}%")
print(f"\n  Kappa reduction (Muon+RB vs Muon):   {(1 - mean_kappa_rb/mean_kappa_muon)*100:+.1f}%")
print(f"  Kappa reduction (Muon+RB vs SGD):     {(1 - mean_kappa_rb/mean_kappa_sgd)*100:+.1f}%")
print(f"  Kappa reduction (Muon vs SGD):         {(1 - mean_kappa_muon/mean_kappa_sgd)*100:+.1f}%")
print(f"\n  Interpretation:")
print(f"    Positive percentages = improvement (lower is better for both metrics)")
print(f"    The Muon+RB vs Muon comparison isolates the effect of global gauge fixing")
print(f"    above and beyond per-layer local gauge fixing.")

---

## 12. Product Kappa Trajectory -- Time-Resolved Conditioning Dynamics

### Reading the Trajectory Table

The table below shows the **seed-averaged** product condition number and loss at selected
training steps. This is the most informative diagnostic in the experiment:

- **Steps 0-50:** Early training. All methods start from the same $\kappa_{\mathrm{prod}}$.
  Watch for when they begin to diverge.
- **Step 50:** First rebalancing event for Muon+RB. Look for a sharp kappa drop.
- **Steps 100-250:** Mid-training. Plain Muon may show $\kappa_{\mathrm{prod}}$ drifting upward
  as cross-layer spectral interference accumulates. Muon+RB should show periodic resets
  (kappa drops every 50 steps). SGD typically shows monotonic kappa growth.
- **Steps 300-500:** Late training. The separation between methods should be maximal.
  If Muon+RB's kappa is substantially lower than Muon's, global gauge fixing is working.

The **loss columns** show whether conditioning improvements translate to optimisation speed.
Better-conditioned networks should converge faster because gradient signals propagate
more faithfully through well-conditioned weight matrices.

In [ ]:
print(f"\n\n{'=' * 85}")
print("PRODUCT KAPPA TRAJECTORY (averaged over seeds)")
print(f"{'=' * 85}")

In [ ]:
# Pad curves to same length
max_len = NUM_STEPS + 1
print(f"\n{'Step':>6} {'SGD kappa':>14} {'Muon kappa':>14} {'Muon+RB kappa':>14} {'SGD loss':>14} {'Muon loss':>14} {'Muon+RB loss':>14}")
print("-" * 95)

In [ ]:
snapshot_steps = [0, 10, 25, 50, 100, 150, 200, 250, 300, 400, 500]
for step_idx in snapshot_steps:
    parts = [f"{step_idx:>6}"]
    for method in ['sgd', 'muon', 'muon_rebalance']:
        kappas = []
        for curve in all_results[method]['kappa_curves']:
            if step_idx < len(curve):
                kappas.append(curve[step_idx])
        mean_k = np.mean(kappas) if kappas else float('nan')
        parts.append(f"{mean_k:>14.1f}")

    for method in ['sgd', 'muon', 'muon_rebalance']:
        losses = []
        for curve in all_results[method]['loss_curves']:
            if step_idx < len(curve):
                losses.append(curve[step_idx])
        mean_l = np.mean(losses) if losses else float('nan')
        parts.append(f"{mean_l:>14.6e}")

    print("".join(parts))

In [ ]:
# --- Intermediate: Kappa trajectory shape analysis ---
print("\n--- Kappa Trajectory Shape Analysis ---")
print("  Characterising the dynamics of each method's product conditioning:\n")
for method, label in [('sgd', 'SGD'), ('muon', 'Muon'), ('muon_rebalance', 'Muon+RB')]:
    all_curves = all_results[method]['kappa_curves']
    # Average across seeds
    min_len = min(len(c) for c in all_curves)
    avg_curve = np.mean([c[:min_len] for c in all_curves], axis=0)
    
    peak_step = int(np.argmax(avg_curve))
    peak_val = avg_curve[peak_step]
    final_val = avg_curve[-1]
    init_val = avg_curve[0]
    
    # Growth factor
    growth = final_val / max(init_val, 1e-15)
    # Monotonicity: what fraction of steps show kappa increasing?
    n_increasing = sum(1 for i in range(1, len(avg_curve)) if avg_curve[i] > avg_curve[i-1])
    mono_pct = n_increasing / (len(avg_curve) - 1) * 100
    
    print(f"  {label:>10}:  init={init_val:.1f}  peak={peak_val:.1f} (step {peak_step})"
          f"  final={final_val:.1f}  growth={growth:.2f}x  increasing {mono_pct:.0f}% of steps")

print(f"\n  Interpretation:")
print(f"    SGD: expect monotonic kappa growth (positive feedback loop)")
print(f"    Muon: expect some kappa drift but slower than SGD (local gauge fixing)")
print(f"    Muon+RB: expect sawtooth pattern with periodic resets every {REBALANCE_EVERY} steps")

---

## 13. Kappa Crossover Analysis -- When Does Conditioning Deteriorate?

### What is a "Kappa Crossover"?

We define a **kappa crossover** as the first step at which an optimizer's product
$\kappa_{\mathrm{prod}}$ exceeds SGD's and **stays above** for at least 80% of all
remaining steps. This is a robust definition that filters out transient fluctuations
while capturing the earliest *durable* conditioning failure.

A crossover is **bad** -- it means the optimizer's product conditioning has become worse
than vanilla SGD's, despite the optimizer's per-step spectral interventions. For plain Muon,
this can happen at depth 8 because per-layer orthogonalisation does not prevent cross-layer
spectral interference from accumulating over hundreds of steps.

### The Hypothesis

**T2 predicts** that Muon+RB should show fewer crossover events than plain Muon, because
the periodic product rebalancing directly resets the cross-layer spectral drift that causes
crossovers.

A value of "None" for the crossover step means: **no crossover occurred** -- the optimizer's
kappa stayed below SGD's for the entire run. This is the ideal outcome.

In [ ]:
print(f"\n\n{'=' * 85}")
print("KAPPA CROSSOVER ANALYSIS")
print(f"{'=' * 85}")

In [ ]:
# For each seed, find when Muon's kappa exceeds SGD's kappa
# and when Muon+RB's kappa exceeds SGD's kappa
muon_crossover_steps = []
muon_rb_crossover_steps = []

In [ ]:
for seed_idx in range(NUM_SEEDS):
    sgd_k = all_results['sgd']['kappa_curves'][seed_idx]
    muon_k = all_results['muon']['kappa_curves'][seed_idx]
    muon_rb_k = all_results['muon_rebalance']['kappa_curves'][seed_idx]

    min_len = min(len(sgd_k), len(muon_k))

    # Find first step where Muon kappa > SGD kappa (sustained)
    muon_cross = None
    for s in range(min_len):
        if muon_k[s] > sgd_k[s]:
            # Check if sustained for 80% of remaining steps
            remaining = min_len - s
            if remaining > 5:
                frac_above = sum(1 for t in range(s, min_len) if muon_k[t] > sgd_k[t]) / remaining
                if frac_above > 0.8:
                    muon_cross = s
                    break
    muon_crossover_steps.append(muon_cross)

    # Same for Muon+RB
    min_len_rb = min(len(sgd_k), len(muon_rb_k))
    muon_rb_cross = None
    for s in range(min_len_rb):
        if muon_rb_k[s] > sgd_k[s]:
            remaining = min_len_rb - s
            if remaining > 5:
                frac_above = sum(1 for t in range(s, min_len_rb) if muon_rb_k[t] > sgd_k[t]) / remaining
                if frac_above > 0.8:
                    muon_rb_cross = s
                    break
    muon_rb_crossover_steps.append(muon_rb_cross)

In [ ]:
print(f"\nPer-seed kappa crossover (Muon overtakes SGD):")
print(f"  {'Seed':>6} {'Muon cross':>12} {'Muon+RB cross':>15}")
print("-" * 40)
for seed_idx in range(NUM_SEEDS):
    mc = muon_crossover_steps[seed_idx]
    mrc = muon_rb_crossover_steps[seed_idx]
    print(f"  {seed_idx:>6} {str(mc):>12} {str(mrc):>15}")

In [ ]:
# Count how many seeds show no crossover for Muon+RB
n_no_cross_muon = sum(1 for x in muon_crossover_steps if x is None)
n_no_cross_rb = sum(1 for x in muon_rb_crossover_steps if x is None)
print(f"\n  Muon:    {n_no_cross_muon}/{NUM_SEEDS} seeds with NO crossover (kappa stays below SGD)")
print(f"  Muon+RB: {n_no_cross_rb}/{NUM_SEEDS} seeds with NO crossover")

In [ ]:
# --- Intermediate: Crossover interpretation ---
print("\n--- Crossover Interpretation ---")
if n_no_cross_rb > n_no_cross_muon:
    print("  Product rebalancing PREVENTED crossovers that plain Muon experienced.")
    print("  This confirms the global gauge-fixing hypothesis: periodic SVD redistribution")
    print("  resets the cross-layer spectral drift before it can exceed SGD's conditioning.")
elif n_no_cross_rb == n_no_cross_muon:
    print("  Both methods showed the same crossover behavior.")
    if n_no_cross_muon == NUM_SEEDS:
        print("  Neither method ever crossed SGD's kappa -- both maintain good conditioning.")
        print("  At this depth, plain Muon may already be sufficient for product conditioning.")
    else:
        print("  Rebalancing did not provide additional crossover protection beyond Muon.")
else:
    print("  UNEXPECTED: Muon+RB showed MORE crossovers than plain Muon.")
    print("  The velocity reset after rebalancing may be causing temporary conditioning spikes.")

# Check if any Muon crossovers occur near rebalancing boundaries
print(f"\n  Note: Rebalancing events occur at steps {REBALANCE_EVERY}, {2*REBALANCE_EVERY}, ..., {(NUM_STEPS//REBALANCE_EVERY)*REBALANCE_EVERY}")
for seed_idx in range(NUM_SEEDS):
    mc = muon_crossover_steps[seed_idx]
    if mc is not None:
        nearest_rebal = round(mc / REBALANCE_EVERY) * REBALANCE_EVERY
        print(f"  Seed {seed_idx}: Muon crossover at step {mc} (nearest rebalance would be step {nearest_rebal})")

---

## 14. Formal Hypothesis Tests

### Test Design Rationale

Four tests evaluate the core hypothesis from complementary angles:

- **T1 (Conditioning):** Does global gauge fixing reduce final product $\kappa$?
  This is the most direct test of the mechanism -- rebalancing explicitly targets
  the product spectrum.

- **T2 (Crossover prevention):** Does global gauge fixing prevent the product
  $\kappa$ from drifting above SGD's? This tests whether rebalancing addresses the
  specific failure mode identified in Exp 2.6.

- **T3 (Convergence vs Muon):** Does better conditioning translate to better
  optimisation? If $\kappa_{\mathrm{prod}}$ is lower but loss is not, the conditioning
  improvement may be cosmetic rather than functional.

- **T4 (Convergence vs SGD):** Does the combined local+global gauge fixing beat
  no gauge fixing at all? This is the baseline sanity check.

**Passing criterion:** Tests 1-3 are the core predictions of the hypothesis.
Test 4 is a baseline check that should almost always pass if Muon itself works.

In [ ]:
print(f"\n\n{'=' * 85}")
print("HYPOTHESIS TESTS")
print(f"{'=' * 85}")

In [ ]:
mean_final_kappa_sgd = np.mean(all_results['sgd']['final_kappas'])
mean_final_kappa_muon = np.mean(all_results['muon']['final_kappas'])
mean_final_kappa_rb = np.mean(all_results['muon_rebalance']['final_kappas'])

In [ ]:
mean_final_loss_sgd = np.mean(all_results['sgd']['final_losses'])
mean_final_loss_muon = np.mean(all_results['muon']['final_losses'])
mean_final_loss_rb = np.mean(all_results['muon_rebalance']['final_losses'])

In [ ]:
# T1: Muon+RB has lower final kappa than plain Muon
t1 = mean_final_kappa_rb < mean_final_kappa_muon
print(f"\nT1: Muon+Rebalance final kappa < plain Muon final kappa?")
print(f"    Muon+RB: {mean_final_kappa_rb:.1f}, Muon: {mean_final_kappa_muon:.1f}")
print(f"    --> {'PASS' if t1 else 'FAIL'}")

In [ ]:
# T2: Muon+RB prevents kappa crossover (in more seeds than plain Muon)
t2 = n_no_cross_rb >= n_no_cross_muon
print(f"\nT2: Muon+RB prevents kappa crossover in >= as many seeds as Muon?")
print(f"    Muon+RB no-cross: {n_no_cross_rb}/{NUM_SEEDS}, Muon: {n_no_cross_muon}/{NUM_SEEDS}")
print(f"    --> {'PASS' if t2 else 'FAIL'}")

In [ ]:
# T3: Muon+RB has better (lower) final loss than plain Muon
t3 = mean_final_loss_rb < mean_final_loss_muon
print(f"\nT3: Muon+Rebalance final loss < plain Muon final loss?")
print(f"    Muon+RB: {mean_final_loss_rb:.6e}, Muon: {mean_final_loss_muon:.6e}")
print(f"    --> {'PASS' if t3 else 'FAIL'}")

In [ ]:
# T4: Muon+RB has lower final loss than SGD
t4 = mean_final_loss_rb < mean_final_loss_sgd
print(f"\nT4: Muon+Rebalance final loss < SGD final loss?")
print(f"    Muon+RB: {mean_final_loss_rb:.6e}, SGD: {mean_final_loss_sgd:.6e}")
print(f"    --> {'PASS' if t4 else 'FAIL'}")

---

## 15. Final Verdict and Interpretation

In [ ]:
total_pass = sum([t1, t2, t3, t4])

### Verdict Decision Logic

- **3-4 tests pass:** CONFIRMED -- Product rebalancing is a viable global gauge-fixing mechanism
  that complements Muon's per-layer orthogonalisation.
- **2 tests pass:** PARTIALLY CONFIRMED -- Some benefit exists but is not comprehensive.
- **0-1 tests pass:** INCONCLUSIVE -- The rebalancing intervention may not be necessary at this
  depth, or the velocity reset cost may outweigh the conditioning benefit.

In [ ]:
print(f"\n\n{'=' * 85}")
print("FINAL VERDICT: PRODUCT-SPACE MUON")
print(f"{'=' * 85}")

In [ ]:
print(f"""
  Tests passed: {total_pass}/4

  Product rebalance every {REBALANCE_EVERY} steps:
    - Final kappa: {mean_final_kappa_rb:.1f} (vs Muon {mean_final_kappa_muon:.1f}, SGD {mean_final_kappa_sgd:.1f})
    - Final loss:  {mean_final_loss_rb:.6e} (vs Muon {mean_final_loss_muon:.6e}, SGD {mean_final_loss_sgd:.6e})
    - Crossover prevention: {n_no_cross_rb}/{NUM_SEEDS} seeds (vs Muon {n_no_cross_muon}/{NUM_SEEDS})
""")

In [ ]:
if total_pass >= 3:
    print("  VERDICT: CONFIRMED -- Product rebalancing controls kappa and improves convergence.")
    print("  The product-space SVD redistribution prevents catastrophic conditioning")
    print(f"  at depth {NUM_LAYERS}.")
elif total_pass >= 2:
    print("  VERDICT: PARTIALLY CONFIRMED -- Some benefit from rebalancing.")
else:
    print("  VERDICT: INCONCLUSIVE -- Product rebalancing did not clearly help.")
    if mean_final_kappa_muon < mean_final_kappa_sgd:
        print("  Note: Plain Muon already controls kappa well at this depth.")

In [ ]:
print("=" * 85)

---

## 16. Conclusions and Broader Implications

### What This Experiment Tests

**Experiment 3.11** probes whether Muon's per-layer gauge-fixing (Newton-Schulz orthogonalisation)
is **sufficient** for controlling the global conditioning of deep linear networks, or whether an
additional **product-space** intervention (SVD rebalancing) is needed.

### The Two Levels of Gauge Fixing

In the Muon-as-RG-Gauge-Fixing framework, weight matrices have:

1. **Local gauge freedom:** The singular-value spread within each $W_\ell$. Muon's per-layer
   orthogonalisation fixes this by projecting gradients onto the orthogonal manifold, ensuring
   each update has unit spectral norm. This keeps $\kappa(W_\ell)$ near 1.

2. **Global gauge freedom:** How the total spectral mass of $W_{\mathrm{prod}}$ is distributed
   across layers. Even with per-layer $\kappa \approx 1$, the product of $L$ rotations can
   amplify certain directions and suppress others. Product rebalancing fixes this by
   SVD-decomposing the full product and redistributing singular values equally.

### Physical Analogy

Think of it like a cascaded optical system. Each layer is a lens. Per-layer gauge fixing
ensures each lens is well-polished (no aberrations). But the lenses can still be **misaligned**
-- their axes of magnification can conspire to create extreme anisotropy in the overall
imaging system. Product rebalancing is like periodically realigning the entire optical train
to distribute magnification uniformly.

### Connection to RG Theory

In the renormalisation-group picture:
- **Per-layer Muon** = gauge fixing at each RG step (local redundancy removal)
- **Product rebalancing** = periodically running the RG flow to a fixed point and
  redistributing the coupling constants (global redundancy removal)
- **The crossover** = the point where local gauge fixing alone fails to prevent
  the accumulation of global gauge redundancy

### Key Findings (from the numerical results above)

Examine the PASS/FAIL outcomes for T1-T4:
- **T1 (kappa reduction):** Does rebalancing achieve lower final $\kappa_{\mathrm{prod}}$?
- **T2 (crossover prevention):** Does rebalancing prevent more crossover events?
- **T3 (loss vs Muon):** Does better conditioning yield faster convergence?
- **T4 (loss vs SGD):** Does the full gauge-fixing pipeline beat no gauge fixing?

### Limitations and Future Directions

- **Linear networks only:** Nonlinear activations introduce additional gauge degrees of
  freedom (e.g., ReLU dead neurons, BatchNorm scale/shift) not captured here.
- **Fixed rebalancing interval $K=50$:** The optimal $K$ likely depends on depth and
  learning rate. An adaptive scheme could rebalance only when $\kappa_{\mathrm{prod}}$
  exceeds a threshold.
- **Velocity reset cost:** Zeroing momentum after rebalancing loses $K$ steps of gradient
  history. A smooth rebalancing scheme that transforms (rather than resets) the velocity
  buffer could reduce this cost.
- **Scaling to larger networks:** The $O(n^3)$ cost of product SVD limits scalability.
  Randomised SVD or hierarchical rebalancing could address this.
- **Comparison with Exp 2.6:** The crossover dynamics here should be compared with the
  depth-dependent crossover scaling found in Exp 2.6 to check consistency.